In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('../../data/raw/banksim.csv')

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().any()

In [ ]:
df[df.duplicated()]

### Sort columns into their data type

In [ ]:
cat_cols, cont_cols = [], []

for col in df.columns:
    if df[col].dtype == "str":
        cat_cols.append(col)
    else:
        cont_cols.append(col)

print("Categorical Columns:", cat_cols)
print("Continuous Columns:", cont_cols)

### Check for columns with no variability
If they have no variance they have no use to a model due to thier features being constant

In [ ]:
cols_to_drop = []
for col in cat_cols:
   unum = df[col].nunique()

   print(f"Unique numbers of {col}s:", unum)

   if unum == 1:
      cols_to_drop.append(col)

print("\nDropping columns due to constant values:", cols_to_drop)

for col in cols_to_drop:
   cat_cols.remove(col)

df.drop(columns=cols_to_drop, inplace=True)

### Extract Customer-based Features (All local features)
These include: 

- "prev_step" - The last time they did a transaction (-1 if no previous transaction exists)
- "time_since_last_transaction" --
- "transaction_count" - Number of transactions

- "amount_sum" - Sum of all previous transactions
- "avg_amount" - Avergae of all past transactions
- "std_amount" - Standard deviation of past transaction amounts
- "amount_zscore" - How many standard deviation this transaction is from past behaviour

- "avg_amount_ratio" - Ratio of how this given transaction compares to customers average
- "log_amount_ratio" - Log transformation to compress extreme values

- "merchant_count" - Number of times a customer has shopped with this merchant
- "category_count" - Number of times a customer has shopped this category
- "prev_merchant_step" - Last time they shopped with this merchant (-1 if they never have)
- "time_since_last_merchant_transaction" --

In [ ]:
# Just checking if the rolling features are working as expected (NEED TO REVERT THIS SORTING BACK TO ORIGINAL ORDER LATER)
df = df.sort_values(["customer", "step"]).copy()
df.rename(columns={"amount": "customer_amount"}, inplace=True)

cust_groups = df.groupby("customer")

# Shift gets the previous step for each customer
df["customer_prev_step"] = cust_groups["step"].shift(1)
df["customer_time_since_last_transaction"] = df["step"] - df["customer_prev_step"]

df["customer_time_since_last_transaction"] = df["customer_time_since_last_transaction"].fillna(-1)
df["customer_prev_step"] = df["customer_prev_step"].fillna(-1)

df["customer_transaction_count"] = cust_groups.cumcount()

df["customer_amount_sum"] = cust_groups["customer_amount"].cumsum() - df["customer_amount"] 
df["customer_amount_sum"] = df["customer_amount_sum"].fillna(0)

df["customer_avg_amount"] = df["customer_amount_sum"] / df["customer_transaction_count"]
df["customer_avg_amount"] = df["customer_avg_amount"].fillna(df["customer_amount"])

# Expanding std gives the std of all previous transactions, shift(1) makes it so that the current transaction is not included in the std calculation
df["customer_std_amount"] = cust_groups["customer_amount"].expanding().std().reset_index(level=0, drop=True).shift(1)
df["customer_std_amount"] = df["customer_std_amount"].fillna(0)

df["customer_avg_amount_ratio"] = df["customer_amount"] / df["customer_avg_amount"]
df["customer_log_amount_ratio"] = np.log1p(df["customer_avg_amount_ratio"])

df["customer_zscore"] = (df["customer_amount"] - df["customer_avg_amount"]) / df["customer_std_amount"]
df["customer_zscore"] = df["customer_zscore"].fillna(0)
df["customer_zscore"] = df["customer_zscore"].replace([np.inf, -np.inf], 0)

df["customer_merchant_count"] = df.groupby(["customer", "merchant"]).cumcount()
df["customer_category_count"] = df.groupby(["customer", "category"]).cumcount()

df["customer_prev_merchant_step"] = df.groupby(["customer", "merchant"])["step"].shift(1)
df["customer_time_since_last_merchant_transaction"] = df["step"] - df["customer_prev_merchant_step"]

df["customer_time_since_last_merchant_transaction"] = df["customer_time_since_last_merchant_transaction"].fillna(-1)
df["customer_prev_merchant_step"] = df["customer_prev_merchant_step"].fillna(-1)

df

In [ ]:
# Sort back to original dataset
df = df.sort_index()
df

### Extract Merchant-based features (ALl local features)
All of these are the same as customer based features execept for:
- "merchant_fraud_count" - How many times has fraud occured for this merchant
- "merchant_fraud_rate" - Rate of fraud up to this transaction

In [ ]:
df = df.sort_values(["merchant", "step"]).copy()
merchant_groups = df.groupby("merchant")

df["merchant_transaction_count"] = merchant_groups.cumcount()

df["merchant_past_amount_sum"] = merchant_groups["customer_amount"].cumsum() - df["customer_amount"]
df["merchant_past_amount_sum"] = df["merchant_past_amount_sum"].fillna(0)

df["merchant_avg_amount"] = df["merchant_past_amount_sum"] / df["merchant_transaction_count"]
df["merchant_avg_amount"] = df["merchant_avg_amount"].fillna(df["customer_amount"])

df["merchant_std_amount"] = merchant_groups["customer_amount"].expanding().std().reset_index(level=0, drop=True).shift(1)
df["merchant_std_amount"] = df["merchant_std_amount"].fillna(0)

df["merchant_avg_amount_ratio"] = df["customer_amount"] / df["merchant_avg_amount"]
df["merchant_log_amount_ratio"] = np.log1p(df["merchant_avg_amount_ratio"])

df["merchant_amount_zscore"] = (df["customer_amount"] - df["merchant_avg_amount"]) / df["merchant_std_amount"]
df["merchant_amount_zscore"] = df["merchant_amount_zscore"].fillna(0)
df["merchant_amount_zscore"] = df["merchant_amount_zscore"].replace([np.inf, -np.inf], 0)

df["merchant_fraud_count"] = merchant_groups["fraud"].cumsum() - df["fraud"]
df["merchant_fraud_rate"] = df["merchant_fraud_count"] / df["merchant_transaction_count"]
df["merchant_fraud_rate"] = df["merchant_fraud_rate"].fillna(0)

df["merchant_prev_step"] = merchant_groups["step"].shift(1)
df["merchant_time_since_last_transaction"] = df["step"] - df["merchant_prev_step"]

df["merchant_time_since_last_transaction"] = df["merchant_time_since_last_transaction"].fillna(-1)
df["merchant_prev_step"] = df["merchant_prev_step"].fillna(-1)

df


In [ ]:
# Sort back to original dataset
df = df.sort_index()
df

### Extract Global Features
Features are the same as both customer and merchant just no longer localised features.

In [ ]:
df["global_transaction_count"] = df.index

df["global_amount_sum"] = df["customer_amount"].cumsum() - df["customer_amount"]
df["global_amount_sum"] = df["global_amount_sum"].fillna(0)

df["global_avg_amount"] = df["global_amount_sum"] / df["global_transaction_count"]
df["global_avg_amount"] = df["global_avg_amount"].fillna(df["customer_amount"])

df["global_amount_ratio"] = df["customer_amount"] / df["global_avg_amount"]
df["global_log_amount_ratio"] = np.log1p(df["global_amount_ratio"])

df["global_std_amount"] = df["customer_amount"].expanding().std().shift(1).reset_index(drop=True)
df["global_std_amount"] = df["global_std_amount"].fillna(0)

df["global_z_score"] = (df["customer_amount"] - df["global_avg_amount"]) / df["global_std_amount"]
df["global_z_score"] = df["global_z_score"].fillna(0)
df["global_z_score"] = df["global_z_score"].replace([np.inf, -np.inf], 0)

df["global_median_amount"] = df["customer_amount"].expanding().median().shift(1).reset_index(drop=True)
df["global_median_amount"] = df["global_median_amount"].fillna(df["customer_amount"])

df["global_median_amount_ratio"] = df["customer_amount"] / df["global_median_amount"]
df["global_log_median_amount_ratio"] = np.log1p(df["global_median_amount_ratio"])

df.drop(columns="global_transaction_count", inplace=True)
df

In [ ]:
df.describe()

### Hot-Encode values
Converts categorical data such as age and data into binary so the model can interpret them

In [ ]:
df = pd.get_dummies(df, columns=["age", "gender"])

# Just incase it writes back as string when reading the dataset
bool_cols = df.select_dtypes(include=["bool"]).columns
df[bool_cols] = df[bool_cols].astype(int)

df

### Reduce Dimensions of the dataset
Drop features for reasons such as:
- Being used to derive other more useful features.
- Better signal for the models.
- Data analysis pointed towards dropping it.

Overall for the base DescisionTree it has reduced False Negatives from 77 -> 47 but raised False Positives from 674 -> 1468,
and for the XGBoost model it raised it from 25 -> 28 but also dropped False Positives from 632 -> 586.

From the permutation and importance graphs of both models a reason for some of these changes such as False Posiitves within the DecisionTree is when I removed some unnesscary global features it was harder for the model to distibguihs rules as local values are more noisy for tree values, XGBoost model imporves this as it uses more local features compared to the global features.

In [ ]:
# Time based columns as they are arbitrary as time grows and time_since_last_transactions captures better meaning
df.drop(columns=["customer_prev_step", "customer_prev_merchant_step", "merchant_prev_step"], inplace=True)

# Sums and averages are captured in the ratios and z-scores, so can drop them to reduce dimensionality
df.drop(columns=["customer_amount_sum", "customer_avg_amount", "merchant_past_amount_sum", "merchant_avg_amount", "global_amount_sum"], inplace=True)

# Was only used to dervive fraud rate, can drop it now
df.drop(columns=["merchant_fraud_count"], inplace=True)

# Dropping from data analyisis, median captures more of the more common transaction amounts, compared to the mean
df.drop(columns=["global_avg_amount", "global_amount_ratio", "global_log_amount_ratio", "global_median_amount"], inplace=True)

# From Permutation graphs log values tend to do better
df.drop(columns=["customer_avg_amount_ratio", "merchant_avg_amount_ratio", "global_median_amount_ratio"], inplace=True)

In [ ]:
os.makedirs("../../data/processed", exist_ok=True)
df.to_csv("../../data/processed/processed_data.csv", index=False)

In [ ]:
# Columns to standardise for non tree based models
scale_cols = [
    "customer_amount",
    "customer_time_since_last_transaction",
    "customer_transaction_count",
    "customer_std_amount",
    "customer_merchant_count",
    "customer_category_count",
    "customer_time_since_last_merchant_transaction",
    "merchant_transaction_count",
    "merchant_std_amount",
    "merchant_time_since_last_transaction",
    "global_std_amount"  
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), scale_cols)
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

### Train/Val/Test split
Creates a train/val/test split of 70/15/15 and standardises and transforms each set

In [ ]:
split_step = df["step"].quantile(0.7)
val_step = df["step"].quantile(0.85)

train_df = df[df["step"] <= split_step]
val_df = df[(df["step"] > split_step) & (df["step"] <= val_step)]
test_df = df[df["step"] > val_step]

train_df.drop(columns=["step"], inplace=True)
val_df.drop(columns=["step"], inplace=True)
test_df.drop(columns=["step"], inplace=True)

train_scaled = preprocessor.fit_transform(train_df)
val_scaled = preprocessor.transform(val_df)
test_scaled = preprocessor.transform(test_df)

cols = preprocessor.get_feature_names_out()

train_scaled = pd.DataFrame(train_scaled, columns=cols)
val_scaled = pd.DataFrame(val_scaled, columns=cols)
test_scaled = pd.DataFrame(test_scaled, columns=cols)

train_scaled

In [ ]:
os.makedirs("../../results/models", exist_ok=True)

os.makedirs("../../data/processed/supervised", exist_ok=True)
os.makedirs("../../data/processed/unsupervised", exist_ok=True)

tree_train_df = train_scaled.copy()
tree_train_df.drop(columns=["customer", "merchant", "category"], inplace=True)
tree_train_df.to_csv("../../data/processed/supervised/train_data.csv", index=False)

tree_val_df = val_scaled.copy()
tree_val_df.drop(columns=["customer", "merchant", "category"], inplace=True)
tree_val_df.to_csv("../../data/processed/supervised/val_data.csv", index=False)

tree_test_df = test_scaled.copy()
tree_test_df.drop(columns=["customer", "merchant", "category"], inplace=True)
tree_test_df.to_csv("../../data/processed/supervised/test_data.csv", index=False)

nn_train_df = train_scaled.copy()
nn_train_df = nn_train_df[nn_train_df["fraud"] == 0]
nn_train_df.to_csv("../../data/processed/unsupervised/train_data.csv", index=False)

nn_val_df = val_scaled.copy()
nn_val_df.to_csv("../../data/processed/unsupervised/val_data.csv", index=False)

nn_test_df = test_scaled.copy()
nn_test_df.to_csv("../../data/processed/unsupervised/test_data.csv", index=False)
